<h1 style="font-size:46px;font-family:'Sour Gummy', cursive; color:#228B22;text-align:center;">
Mushroom Classification: To Eat or Not To Eat 
</h1>

<h1 style="font-size:16px; color:#410200;"><strong>Overview:</strong> The goal of this notebook is to build a machine learning model that predicts if a mushroom is edible or poisonous based on physical characteristics. The dataset contains categorical mushroom features, so the features must be one-hot encoded before modeling to turn target responses into numerical values.</h1>

#### *This notebook follows these major steps:* ####

    1). Load the mushroom dataset and assign proper column headers using the '.names' file. 
    2). Split the data into training and testing sets using an 80/20 split. 
    3). One-hot encode the categorical feature variables.
    4). Label encode the response variable. 
    5). Train a basic sequential neural network using the encoded data. 
    6). Evaluate the model using a confusion matrix
    7). Apply PCA to reduce dimensionality while keeping 95% of the variance. 
    8). Train a second neural network using the PCA-transformed data. 
    9). Compare model performance and training time, 
    10). Save the PCA neural network model.

In [ ]:
# Common imports 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

# Scikit-learn imports 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Tensorflow/keras imports 
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Input 

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 Data Loading
</h2>

<p style="font-size:15px; color:#410200;">
The mushroom dataset is loaded into a pandas DataFrame. Because the raw data file does not include columns headers, the feature names from the '.names' file are manually assigned. The first column 'class' is the target variable and indicates whether each mushroom is edible ('e') or poisonous (p')
</p>

In [ ]:
# Column names based on the mushroom dataset .name file 
column_names = [
    "class", 
    "cap_chape", 
    "cap_surface", 
    "cap_color", 
    "bruises", 
    "odor", 
    "gill_attachment", 
    "gill_spacing", 
    "gill_size", 
    "gill_color", 
    "stalk_shape", 
    "stalk_root", 
    "stalk_surface_above_ring", 
    "stalk_surface_below_ring", 
    "stalk_color_above_ring", 
    "stalk_color_below_ring", 
    "veil_type", 
    "veil_color",
    "ring_number", 
    "ring_type", 
    "spore_print_color",
    "population", 
    "habitat"
]

# Load the dataset 
mushroom_df = pd.read_csv("C:/Users/alyss/Downloads/agaricus-lepiota.csv", header=None, names=column_names)

# Display the first few rows 
mushroom_df.head()

In [ ]:
# Check the shape of the dataset 
print("Dataset shape:", mushroom_df.shape)

# Check for missing values or unusual placeholders
mushroom_df.info()

# View class distribution 
mushroom_df["class"].value_counts()

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 Feature and Target Seperation
</h2>

<p style="font-size:15px; color:#410200;">
The dataset is divided into feature variables, X, and the target variable, y. The 'class' column is removed from the features and stored seperately as the response variable.
</p>

In [ ]:
# Separate the response variable from the feature vairables
X= mushroom_df.drop("class", axis=1)
y= mushroom_df["class"]

print("Original number of features before encoding:", X.shape[1])

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 Train/Test Split
</h2>

<p style="font-size:15px; color:#410200;">
The dataset is split into training and testing sets using an 80/20 ratio. This ensures the model is trained on one portion of the data and is evaluated on unseen data. Stratification is used to preserve the class distribution. 
</p>

In [ ]:
# Split the data into 80% training and 20% testing 
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20,
    random_state=42, 
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 Data Preprocessing
</h2>

<p style="font-size:15px; color:#410200;">
The dataset contains categorial features that must be converted into numerical form for use in neural network. One-Hot Encoding is applied to the feature variables to create binary columns for each category. The target variable is encoded using LabelEncoder, where 'e' (edible) is mapped to 0 and 'p' (poisonous) is mapped to 1.
</p>

In [ ]:
# One-hot encode the categorical feature variables 
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

# Label encode the response variable 
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Encoded training feature shape:", X_train_encoded.shape)
print("Encoded testing feature shape:", X_test_encoded.shape)

print("Label encoding classes:", label_encoder.classes_)

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 Model Creation
</h2>

<p style="font-size:15px; color:#410200;">
A simple neural network is constructed for binary classification. The model consists of an input layer that matches the number of encoded features and an output layer with a single neuron using a sigmoid activation function to produce probability values.
</p>

In [ ]:
# Save the number of encoded features 
number_of_encoded_features = X_train_encoded.shape[1]

# Build the original neural network model 
original_model = Sequential([
    Input(shape=(number_of_encoded_features,)), 
    Dense(1, activation="sigmoid")
])

# Compile the model 
original_model.compile(
    optimizer="adam", 
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display model summary 
original_model.summary()

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 Model Training 
</h2>

<p style="font-size:15px; color:#410200;">
The neural network is trained using the training data. The %%time magic command is used to measure how lonhg hthe training process takes.
</p>

In [ ]:
%%time 

# Train the original neural network 
original_history = original_model.fit(
    X_train_encoded, 
    y_train_encoded, 
    epochs=25, 
    batch_size=32, 
    validation_split=0.20, 
    verbose=1
)

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 Model Evaluation
</h2>

<p style="font-size:15px; color:#410200;">
The trained model is evaluated using the testing data. Predicted probabilities are converted into class labels using a threshold of 0.5. A confusion matrix is used to visualize the model's performance and identify correct and incorrect classifications. 
</p>

In [ ]:
# Generate predicted probabilities 
original_probabilities = original_model.predict(X_test_encoded)

# Convert probabilities to class predictions
original_predictions = (original_probabilities >= 0.5).astype(int).flatten()

# Display confusion matrix 
ConfusionMatrixDisplay.from_predictions(
    y_test_encoded, 
    original_predictions,
    display_labels=label_encoder.classes_
)

plt.title("Original Neural Network Confusion Matrix")
plt.show()

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 PCA Dimensionality Reduction 
</h2>

<p style="font-size:15px; color:#410200;">
Principal Component Analysis (PCA) is applied to reduce the number of input features while retaining 95% of the varience in the data. This reduces complexity and may imporve training efficiency. A new model will be trained using this transformed dataset.
</p>

In [ ]:
# Create PCA model that keeps 95% of the vairence 
pca= PCA(n_components=0.95, random_state=42)

# Fit PCA on the encoded training data and transform both train and test data 
X_train_pca = pca.fit_transform(X_train_encoded)
X_test_pca =pca.transform(X_test_encoded)

print("Original encoded feature count:", X_train_encoded.shape[1])
print("PCA feature count:", X_train_pca.shape[1])
print("Variance retained:", np.sum(pca.explained_variance_ratio_))

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 PCA Model Creation
</h2>

<p style="font-size:15px; color:#410200;">
A second neural network is created using the PCA-Transformed data. The structure is similar to the original model, but the input layer size reflects the reduced number of features.
</p>

In [ ]:
# Save the number of PCA features 
number_of_pca_features = X_train_pca.shape[1]

# Build the PCA neural network model 
pca_model = Sequential([
    Input(shape=(number_of_pca_features,)), 
    Dense(1, activation="sigmoid")
])

# Compile the PCA model 
pca_model.compile(
    optimizer="adam", 
    loss="binary_crossentropy", 
    metrics=["accuracy"]
)

# Display model summary 
pca_model.summary()

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 PCA Model Training
</h2>

<p style="font-size:15px; color:#410200;">
The PCA-based neural network is trained using the reduced feature set. Training time is recorded for comparison with the original model. 
</p>

In [ ]:
%%time

# Train the PCA neural network 
pca_history = pca_model.fit(
    X_train_pca, 
    y_train_encoded, 
    epochs=25, 
    batch_size=32,
    validation_split=0.20, 
    verbose=1
)

<h2 style="color:#6A0DAD; font-size:20px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🍄 PCA Model Evaluation
</h2>

<p style="font-size:15px; color:#410200;">
The PCA model is evaluated using a confusion matrix, as well. Its performance is compared to the original model to assess the impact of dimensionality reduction. 
</p>

In [ ]:
# Generate predicted probabilities from PCA model 
pca_probabilities = pca_model.predict(X_test_pca)

# Convert probabilities to class predictions
pca_predictions = (pca_probabilities >= 0.5).astype(int).flatten()

# Display confusion matrix 
ConfusionMatrixDisplay.from_predictions(
    y_test_encoded, 
    pca_predictions,
    display_labels = label_encoder.classes_
)

plt.title("PCA Neural Netowrk Confusion Matrix")
plt.show()

In [ ]:
def display_mushroom_prediction(prediction_value):
    """
    Displays a mushroom prediction using color-coded test. 
    Red is used for poisonous predictions. 
    Green is used for edible predictions.
    """

    predicted_label = label_encoder.inverse_transform([prediction_value])[0]

    if predicted_label == 'p':
        color="red"
        message = "The model predicts that this mushroom is POISONOUS."
    else:
        color = "green"
        message = "The model predicts that this mushroom is EDIBLE."

    display_html = f"""
    <div style="
        background-color:{color};
        color:white;
        padding:15px;
        border-radius:10px;
        font-size:20px;
        font-weight:bold;
        text-align:center;">
        {message}
    </div>
    """

    from IPython.display import HTML, display
    display(HTML(display_html))

In [ ]:
# Display the first predictions from the PCA model 
display_mushroom_prediction(pca_predictions[0])

In [ ]:
# Locate a prediction that is edible to verify color coding worked 
display_mushroom_prediction(pca_predictions[2])

<h2 style="color:#6A0DAD; font-size:26px; border-bottom:2px solid #2E7D32; padding-bottom:5px;">
🧠 Model Design & Feature Engineering Insights 
</h2>

### Feature Expansion from One-Hot Encoding

The original dataset contained 22 categorical features. After applying One-Hot Encoding, the feature space expanded to 116 features, as each categorical value was converted into its own binary column.

While this transformation allows models to better interpret categorical data, it significantly increases dimensionality, which can impact computational efficiency and model complexity.

---

### Neural Network Output Design

The output layer of the neural network uses a single unit. This is appropriate for a binary classification problem, as the model outputs a single probability representing the likelihood that a mushroom belongs to one of the two classes (edible or poisonous).

---

### Model Complexity and Parameters

The neural network architecture connects each input feature to a single output neuron. With 116 input features and one output unit, the model contains 116 input-to-output connections plus one bias term, resulting in a total of 117 trainable parameters.

---

### Dimensionality Reduction with PCA

Principal Component Analysis (PCA) was applied to reduce the dimensionality of the dataset while retaining 95% of the variance. This reduced the feature space from 116 features to 40 principal components.

This reduction simplifies the input space while preserving most of the information contained in the original encoded dataset.

---

### Impact on Model Architecture

Because PCA transforms the dataset into a reduced set of features, the input dimension of the neural network must be adjusted accordingly. The model using PCA operates on 40 input features instead of the original 116, requiring modification of the input layer configuration.

---

### Training Time Comparison

The original neural network trained in approximately 16.3 seconds, while the PCA-based model trained in approximately 17.3 seconds. Although dimensionality reduction is often expected to improve computational efficiency, this was not observed in this case.

This outcome can be attributed to several factors:

- PCA produces dense feature representations, which may increase computational cost  
- The original neural network was already relatively simple  
- The difference in feature count was not large enough to significantly impact performance  
- Minor variations in system performance can affect timing results  

This highlights that dimensionality reduction does not always lead to faster training and should be evaluated within the context of the specific model and dataset.

In [ ]:
# Exporting the PCA model 
pca_model.save("mushroom_pca_model.h5")